# From Effects to Decisions
## P@S Lecture 18: You estimated the effect. Now what?

Coppock, Hill & Vavreck (2020) ran 59 weekly experiments during the 2016
presidential campaign and found that political ads have small effects.
Today: replicate their key findings, then explore what this means for
campaign strategy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

BLUE, RED, GREEN, GRAY, ORANGE = '#2980b9', '#c0392b', '#27ae60', '#7f8c8d', '#e67e22'

## Section 1: Synthetic data matching Coppock et al. (2020)

We generate 59 experiments, ~600 participants each, with ATEs ~0.05 on a 1-5 scale.
Real replication data: Harvard Dataverse, DOI 10.7910/DVN/TN7KWR.

In [ ]:
n_experiments = 59
experiments = []
for i in range(n_experiments):
    n = np.random.randint(400, 800)
    true_ate = np.random.normal(0.05, 0.04)
    ad_party = 'Democratic' if i % 2 == 0 else 'Republican'
    for j in range(n):
        t = np.random.binomial(1, 0.5)
        y = 3.0 + np.random.normal(0, 0.8) + t * true_ate + np.random.normal(0, 0.5)
        y = np.clip(y, 1, 5)
        r = np.random.random()
        rp = 'Democrat' if r < 0.4 else ('Republican' if r < 0.7 else 'Independent')
        experiments.append({'experiment': i+1, 'treatment': t, 'outcome': y,
                           'ad_party': ad_party, 'respondent_party': rp})

df = pd.DataFrame(experiments)
print(f"Observations: {len(df):,} | Experiments: {n_experiments} | Avg N: {len(df)//n_experiments}")

In [ ]:
# Compute ATE and 95% CI for each experiment
results = []
for exp_id in range(1, n_experiments + 1):
    d = df[df['experiment'] == exp_id]
    tr, ct = d[d['treatment']==1]['outcome'], d[d['treatment']==0]['outcome']
    ate = tr.mean() - ct.mean()
    se = np.sqrt(tr.var()/len(tr) + ct.var()/len(ct))
    results.append({'experiment': exp_id, 'ate': ate, 'se': se,
                    'ci_lo': ate - 1.96*se, 'ci_hi': ate + 1.96*se,
                    'ad_party': d['ad_party'].iloc[0]})
res = pd.DataFrame(results)

# Meta-analytic average
w = 1 / res['se']**2
meta_ate = np.average(res['ate'], weights=w)
meta_se = np.sqrt(1 / w.sum())
print(f"Meta-analytic ATE: {meta_ate:.4f} (SE={meta_se:.4f}), p={2*(1-stats.norm.cdf(abs(meta_ate/meta_se))):.4f}")

## Section 2: The forest plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 14))
res_s = res.sort_values('ate')
for i, (_, r) in enumerate(res_s.iterrows()):
    c = BLUE if r['ad_party'] == 'Democratic' else RED
    ax.plot([r['ci_lo'], r['ci_hi']], [i, i], color=c, linewidth=1, alpha=0.6)
    ax.plot(r['ate'], i, 'o', color=c, markersize=4)
dy = len(res_s) + 1
ax.plot(meta_ate, dy, 'D', color='black', markersize=10, zorder=5)
ax.plot([meta_ate-1.96*meta_se, meta_ate+1.96*meta_se], [dy, dy], 'k-', lw=2)
ax.axvline(0, color=GRAY, ls='--', alpha=0.5)
ax.set_xlabel('Average Treatment Effect (favorability, 1-5 scale)')
ax.set_title('59 Experiments: Every Effect Is Small\n(Blue=Dem ads, Red=Rep ads, Diamond=meta-analytic average)')
ax.set_yticks([])
ax.annotate(f'Overall: {meta_ate:.3f}\n(significant but tiny)', xy=(meta_ate, dy),
            xytext=(0.15, dy-5), fontsize=10, arrowprops=dict(arrowstyle='->'))
plt.tight_layout()
plt.show()

**What you should see:** 59 effects, all tiny, clustered near zero.
Most confidence intervals cross zero. The overall average (diamond)
is about 0.05, statistically significant but practically meaningless.

## Section 3: Does it matter who sees the ad?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for j, party in enumerate(['Democrat', 'Republican', 'Independent']):
    sub = df[df['respondent_party'] == party]
    tr, ct = sub[sub['treatment']==1]['outcome'], sub[sub['treatment']==0]['outcome']
    ate = tr.mean() - ct.mean()
    se = np.sqrt(tr.var()/len(tr) + ct.var()/len(ct))
    ax.bar(j, ate, yerr=1.96*se, color=[BLUE, RED, GRAY][j], capsize=10, width=0.5)
ax.set_xticks([0,1,2]); ax.set_xticklabels(['Democrat','Republican','Independent'])
ax.axhline(0, color='k', lw=0.5)
ax.set_ylabel('ATE'); ax.set_title('Ad Effects by Respondent Party\n(Error bars = 95% CI)')
ax.set_ylim(-0.1, 0.15)
plt.tight_layout(); plt.show()

**What you should see:** Three bars of roughly equal (tiny) height.
No subgroup where ads produce meaningfully large effects.

## Section 4: Simulating heterogeneous treatment effects (GOTV)

In [ ]:
n_v = 10000
voters = pd.DataFrame({
    'age': np.random.uniform(18, 80, n_v),
    'past_votes': np.random.poisson(3, n_v),
    'treatment': np.random.binomial(1, 0.5, n_v),
})
voters['true_tau'] = 0.15 * np.exp(-voters['age']/40) * np.exp(-voters['past_votes']/3)
voters['base_prob'] = (0.3 + 0.005*voters['age'] + 0.08*voters['past_votes']).clip(0.1, 0.95)
voters['vote_prob'] = (voters['base_prob'] + voters['treatment']*voters['true_tau']).clip(0.01, 0.99)
voters['voted'] = np.random.binomial(1, voters['vote_prob'])

ate = voters[voters['treatment']==1]['voted'].mean() - voters[voters['treatment']==0]['voted'].mean()
print(f"Overall ATE: {ate:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, lbl, clr in [(ax1, 'age', 'Age Group', GREEN), (ax2, 'past_votes', 'Past Elections', BLUE)]:
    bins = pd.qcut(voters[col], 4)
    effs = voters.groupby([bins, 'treatment'])['voted'].mean().unstack()
    effs['ate'] = effs[1] - effs[0]
    ax.bar(range(4), effs['ate'].values, color=clr, width=0.5)
    ax.set_xticks(range(4)); ax.set_xticklabels([str(x) for x in effs.index], fontsize=8)
    ax.set_xlabel(lbl); ax.set_ylabel('Treatment Effect')
    ax.set_title(f'Who Responds to Contact?\nBy {lbl}')
plt.suptitle('Heterogeneous Treatment Effects', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

**What you should see:** Effects are largest for young, infrequent voters.
Habitual voters barely respond. This is the Imai & Strauss insight.

## Section 5: Causal vs. predictive targeting

In [ ]:
n_target = n_v // 10
pred_idx = voters.nsmallest(n_target, 'base_prob').index  # Least likely to vote
causal_idx = voters.nlargest(n_target, 'true_tau').index   # Most responsive
overlap = len(set(pred_idx) & set(causal_idx))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.scatter(voters['base_prob'], voters['true_tau'], alpha=0.05, s=5, color=GRAY)
ax1.scatter(voters.loc[pred_idx, 'base_prob'], voters.loc[pred_idx, 'true_tau'], alpha=0.3, s=15, color=RED, label='Predictive')
ax1.scatter(voters.loc[causal_idx, 'base_prob'], voters.loc[causal_idx, 'true_tau'], alpha=0.3, s=15, color=GREEN, label='Causal')
ax1.set_xlabel('Predicted Turnout (Y-hat)'); ax1.set_ylabel('Treatment Effect (tau-hat)')
ax1.set_title(f'Two Strategies, Different People\n(Overlap: {overlap/n_target:.0%})'); ax1.legend()

pred_votes = voters.loc[pred_idx, 'true_tau'].sum()
causal_votes = voters.loc[causal_idx, 'true_tau'].sum()
ax2.bar(['Predictive\n(unlikely voters)', 'Causal\n(responsive voters)'],
        [pred_votes, causal_votes], color=[RED, GREEN], width=0.4)
ax2.set_ylabel('Expected Additional Votes')
ax2.set_title(f'Causal Targeting: {causal_votes/pred_votes:.1f}x More Votes')
plt.tight_layout(); plt.show()

**What you should see:** The two strategies select DIFFERENT people.
Causal targeting produces significantly more additional votes because
it targets people who RESPOND to contact, not just people unlikely to vote.

## Key Takeaways

1. **Ad effects are small.** 59 experiments, ~0.05 on a 5-point scale.
2. **Subgroups don't help.** Small for Democrats, Republicans, Independents.
3. **Treatment effects are heterogeneous.** Young, infrequent voters respond most.
4. **Causal targeting beats predictive targeting.** Target responsiveness, not need.